# 09 - Multi-PDF Loading

## Purpose

This notebook extends the OpenAI-based PDF RAG chatbot from one PDF to multiple PDFs.

Instead of loading a single PDF file, this notebook loads every PDF inside a Databricks Volume folder and converts all pages into LangChain `Document` objects.

Each page keeps useful metadata such as:

- Source file name
- Source path
- Page number
- Document type

This metadata will later allow the chatbot to answer questions across multiple PDFs and show which PDF and page were used as sources.

## Input

PDF folder path:

```text
/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs


Notebook Flow

Multi-PDF folder
        ↓
Find all PDF files
        ↓
Load each PDF with PyPDFLoader
        ↓
Add metadata to every page
        ↓
Combine all pages into one document list
        ↓
Preview loaded documents

In [0]:
%pip install langchain langchain-community langchain-openai pypdf tiktoken chromadb

In [0]:
pdf_folder_path = "/Volumes/workspace/365pdf/365pdfupload/multi_pdf_docs"

print("PDF folder path:")
print(pdf_folder_path)

In [0]:
files_info = dbutils.fs.ls(pdf_folder_path)

print("Files found in folder:")

for file_info in files_info:
    print(file_info.path)

In [0]:
pdf_files = [
    file_info.path.replace("dbfs:", "")
    for file_info in files_info
    if file_info.path.lower().endswith(".pdf")
]

print("Number of PDF files found:", len(pdf_files))

for pdf_file in pdf_files:
    print(pdf_file)

In [0]:
import os
from pypdf import PdfReader
from langchain_core.documents import Document

def normalize_volume_path(path):
    """
    Converts dbfs:/Volumes/... paths into /Volumes/... paths
    because pypdf works better with local-style Volume paths.
    """
    if path.startswith("dbfs:"):
        return path.replace("dbfs:", "")
    return path


all_pdf_docs = []

for pdf_file in pdf_files:
    pdf_file = normalize_volume_path(pdf_file)

    print("Loading PDF:")
    print(pdf_file)

    file_name = os.path.basename(pdf_file)

    reader = PdfReader(pdf_file)

    for page_index, page in enumerate(reader.pages):
        page_text = page.extract_text()

        if page_text and page_text.strip():
            doc = Document(
                page_content=page_text,
                metadata={
                    "source_file": file_name,
                    "source_path": pdf_file,
                    "page_number": page_index,
                    "document_type": "pdf"
                }
            )

            all_pdf_docs.append(doc)

print("All PDFs loaded successfully")
print("Total page documents loaded:", len(all_pdf_docs))

In [0]:
for i, doc in enumerate(all_pdf_docs[:5]):
    print(f"\nDocument {i}")
    print("Metadata:")
    print(doc.metadata)
    print()
    print("Content preview:")
    print(doc.page_content[:700])
    print("-" * 80)

In [0]:
from collections import Counter

file_page_counts = Counter(
    doc.metadata.get("source_file")
    for doc in all_pdf_docs
)

print("PDF loading summary:")
print("-" * 80)

for file_name, page_count in file_page_counts.items():
    print(f"{file_name}: {page_count} pages")

In [0]:
if len(pdf_files) == 0:
    raise ValueError("No PDF files found. Check the folder path.")

if len(all_pdf_docs) == 0:
    raise ValueError("PDF files were found, but no pages were loaded.")

print("Validation passed")
print("PDF files:", len(pdf_files))
print("Loaded page documents:", len(all_pdf_docs))